In [1]:
import warnings
import glob
import os
import pickle
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils.plots import *
from utils.data import *
from biophysical_model.dopamine_toolbox import DrugSimulations,get_delta_fr,initialize_config_drugs
from utils.path import PathConfig

warnings.filterwarnings("ignore")
rseed =10
np.random.seed(rseed)

## Reproduction of drug manipulations

- This code plots the results from the biophysical simulations performed by adding an additional D2 receptor agonist (bromocriptine)
- The simulations were done to reproduce the effects of bromocriptine on reversal learning in the study: **Cools, R., Frank M.J., Gibbs S., Miyakawa A., Jagust W. & D'Esposito M.  Striatal dopamine predicts outcome-specific reversal learning and its sensitivity to dopaminergic drug administration. J. Neurosci. 29, 1538–1543 (2009)** in which healthy humans were given this drug and asked to perform a reversal task
- The output of this notebook are the files used in the notebook `plot_drugs_from_biophysical_simulations`


In [2]:
choose_drug = 'bromocriptine'
paths = PathConfig()
g_dir = paths.g_dir
save_dir = 'data/local/analysis/biophysical_model/drug_experiments'
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)
data_path = os.path.join(g_dir, 'raw_data','control')
file_list = glob.glob(data_path + '/*.pickle')

Load single neuron data and compute spontaneous activity for unrewarded trials

Outputs:
- `sp_mats`: list of length= N neurons, where each element is an array of size: N trials x N time bins
- `time_alls`:list of length= N neurons, where each element is a vector of size: N time bins

In [3]:
sec_base = 5
window_re = [-sec_base*10000, 10000]
dt = 0.01
sp_mats,time_alls = [],[]
for iff in tqdm(np.arange(len(file_list)),'neuron:'):
    with open(file_list[iff], 'rb') as handle:
        unit = pickle.load(handle)
    sp_times = unit['data']['responses']['spike'] 
    event_re = unit['data']['events']['odorOn']
    trial_types = unit['data']['TrialTypes']
    trials_unrew =  np.concatenate((np.argwhere(trial_types==2),np.argwhere(trial_types==4)))
    rate_mat = []
    for icound,itr in zip(range(len(trials_unrew)),trials_unrew):
        event_t = event_re[itr]
        window_indeces =  np.intersect1d(np.argwhere((sp_times > (event_t + window_re[0])) ),np.argwhere((sp_times < (event_t + window_re[1])) ))    
        sp_times_unit =  np.divide(sp_times[window_indeces],1000) # to secs 
        if len(sp_times_unit)>15:
            sp_times_unit = sp_times_unit-sp_times_unit[0]+dt
            _, rate_base, _, _ = spikes_to_rate(dt, sp_times_unit)
            if len(rate_base)>500+sec_base*10000/10:
                rate_mat.append(rate_base)
    len_ = np.min([len(uu) for uu in rate_mat])
    mat = np.asarray([uu[0:len_] for uu in rate_mat])
    tall = dt*np.arange(mat.shape[-1]) + 0.5*dt
    sp_mats.append(mat)
    time_alls.append(tall)


neuron::   0%|          | 0/44 [00:00<?, ?it/s]

neuron:: 100%|██████████| 44/44 [00:04<00:00,  9.16it/s]


### Run biophysical simulations

The simulations are performed as in the notebook `biophysical_simulations_from_data` with the addition of a D2 receptor agonist **bromocriptine**

We performed a  sweep of the following parameters:

- **Drug concentration**: range from $10^{-1.5}$ to $10^{2}$ nM  
- **Drug efficacy:** range fom 0.1 to 0.6 efficacy on both D2s and D2l receptors
- **Dopamine baseline levels:** this was obtained by inducing a constant shift in the spontaneous activity of the neurons obtained above. The shifts were done to 6 different baseline firing rates levels. 


In [11]:
drug_concv, efficacies, _, _, _,_,_,_,_,_,_ = initialize_config_drugs()
for id_unit in range(len(file_list)):
    print('Unit : ', id_unit, ' of ', len(file_list))
    sp_mat = sp_mats[id_unit]
    time_all = time_alls[id_unit]
    fr_mats,delta_fr =  get_delta_fr(sp_mat,time_all,sec_base=25,dt=0.01)
    sim = DrugSimulations(drug = 'bromocriptine',efficacies=efficacies,drug_concentrations=drug_concv,delta_fr=delta_fr,n_steps=fr_mats.shape[-1],dt=dt)
    sim.run_simulations(fr_mats)
    res = sim.res
    fname = file_list[id_unit].split('/')[-1].split('.pickle')[0]
    np.save(os.path.join(save_dir,fname + choose_drug + '_results_only_base.npy'),res)


Unit :  0  of  44


DrugConcentration:   0%|          | 0/10 [00:00<?, ?it/s]

DrugConcentration: 100%|██████████| 10/10 [00:50<00:00,  5.07s/it]


Unit :  1  of  44


DrugConcentration: 100%|██████████| 10/10 [00:51<00:00,  5.12s/it]


...

DrugConcentration: 100%|██████████| 10/10 [00:48<00:00,  4.81s/it]


Unit :  14  of  44


DrugConcentration: 100%|██████████| 10/10 [00:51<00:00,  5.14s/it]


Unit :  15  of  44


DrugConcentration:  20%|██        | 2/10 [00:10<00:40,  5.03s/it]